# 🩺 Prescription NER — Bio_ClinicalBERT Fine-Tuning
**Model:** `emilyalsentzer/Bio_ClinicalBERT`  
**Task:** Named Entity Recognition on prescription data  

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your `Prescription_Reader` folder to Google Drive (`My Drive/Prescription_Reader`)
3. Run cells top to bottom

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ── Cell 2: Set project directory ────────────────────────────────────────────
import os

PROJECT_PATH = '/content/drive/MyDrive/Prescription_Reader'
os.chdir(PROJECT_PATH)
print('📁 Working directory:', os.getcwd())
print('📂 Contents:', os.listdir('.'))

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers torch tqdm python-dotenv seqeval
print('✅ Dependencies installed')

In [ ]:
# ── Cell 4: Verify GPU ────────────────────────────────────────────────────────
import torch

gpu_available = torch.cuda.is_available()
print('GPU available :', gpu_available)
if gpu_available:
    print('GPU name      :', torch.cuda.get_device_name(0))
    print('VRAM (GB)     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 5: Verify training data ──────────────────────────────────────────────
from pathlib import Path

train_file = Path('data/custom_indian/train.conll')
dev_file   = Path('data/custom_indian/dev.conll')

print('train.conll exists:', train_file.exists(), '|', sum(1 for _ in open(train_file)) if train_file.exists() else 'MISSING', 'lines')
print('dev.conll exists  :', dev_file.exists(),   '|', sum(1 for _ in open(dev_file))   if dev_file.exists()   else 'MISSING', 'lines')

label_map = Path('models/label_map.json')
print('label_map.json    :', label_map.exists())

In [ ]:
# ── Cell 6: (Optional) Adjust hyperparameters before training ─────────────────
# Uncomment and edit if you want to override config.py values at runtime

import sys
sys.path.insert(0, '.')

from src.utils.config import get_model_config
cfg = get_model_config()

# cfg.num_epochs  = 20    # increase if val_loss is still dropping
# cfg.batch_size  = 16    # reduce if you get CUDA out-of-memory errors
# cfg.fp16        = True  # keep True for GPU, set False for CPU

print('Model            :', cfg.pretrained_model_name)
print('Epochs           :', cfg.num_epochs)
print('Batch size       :', cfg.batch_size)
print('Learning rate    :', cfg.learning_rate)
print('fp16             :', cfg.fp16)
print('Max seq length   :', cfg.max_length)

In [ ]:
# ── Cell 7: Train the model ───────────────────────────────────────────────────
# This will take ~20-40 min on T4 GPU for 15 epochs
# Model is saved to: models/biobert_prescription/ (inside your Drive folder)

!python retrain.py

In [ ]:
# ── Cell 8: Test inference ────────────────────────────────────────────────────
!python test_ner.py

In [ ]:
# ── Cell 9: (Optional) Re-train from scratch ──────────────────────────────────
# Run this cell if you want to wipe the existing model and retrain
import shutil
from pathlib import Path

model_dir = Path('models/biobert_prescription')
if model_dir.exists():
    shutil.rmtree(model_dir)
    print('🗑️  Old model removed')

!python retrain.py

In [ ]:
# ── Cell 10: (Optional) Download model to your PC ─────────────────────────────
# Skip this if using Drive — model already synced to My Drive/Prescription_Reader/models/
# Use this ONLY if you want to download directly from Colab session storage

import shutil
from google.colab import files

shutil.make_archive('/content/biobert_prescription', 'zip',
                    'models/biobert_prescription')
files.download('/content/biobert_prescription.zip')
print('⬇️  Download started — extract to models/biobert_prescription/ locally')

## 📋 After Training

Your trained model is saved at:
```
My Drive/Prescription_Reader/models/biobert_prescription/
```

To use it **locally on Windows**, download the `biobert_prescription` folder from Drive and place it at:
```
C:\Users\HP\Desktop\Prescription_Reader\models\biobert_prescription\
```
Then run `python test_ner.py` locally.